# Stage 5 - Burning-activity calendar and backtests

**NASA Space Apps 2026 - MODIS/VIIRS hotspot harmonization**

Seasonal climatology + anomaly baselines, then persistence and seasonal-naive forecasters with the PatchTST candidate (auto-skipped with a warning when PyTorch is unavailable), all evaluated with leakage-guarded rolling-origin backtests before the final 14-day probabilistic forecast artifact.

| | |
|---|---|
| Stage | 5 of 6 - offline compute |
| Network required | NO |
| Runs | the committed CLI scripts (`scripts/`) - logic stays out of cells |
| Writes to | `out/calendar/ (forecaster artifact, backtest reports)` |

Rules that make this notebook reproducible (docs/DEVELOPMENT.md): record the
environment below, keep every output under the working directory, never fetch
from PyPI/GitHub/Hugging Face/REST APIs during offline compute, and keep
generated weights/datasets out of Git.


## 0. Environment

In [ ]:
import sys, platform, subprocess
from pathlib import Path

# Kaggle working dir is the ONLY writable location during offline compute.
KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "out"
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd()

print("python   :", sys.version)
print("platform :", platform.platform())
print("machine  :", platform.machine())
print("repo     :", REPO_ROOT)
print("work dir :", WORK_DIR)

RECORD = {"python": sys.version.split()[0], "platform": platform.platform(), "machine": platform.machine()}


train_out = WORK_DIR / "harmonization"   # produced by stage 4
daily_csv = train_out / "daily_aoi_records.csv"
print("daily records:", daily_csv, "| exists:", daily_csv.exists())


## 1. Backtest baselines and the PatchTST candidate

Rolling-origin backtests train strictly on history `<= origin - 1 day` (leakage-guarded, unit-tested). Without torch the candidate is skipped with a clear warning; the baselines remain fully usable.

In [ ]:
import json, hashlib
from pathlib import Path

def run_cli(script: str, *args: str, expect: int = 0) -> subprocess.CompletedProcess:
    """Run a committed CLI script; fail the notebook loudly on bad exits."""
    cmd = [sys.executable, str(REPO_ROOT / "scripts" / script), *args]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=7200, cwd=str(REPO_ROOT))
    if proc.returncode != expect:
        print(proc.stdout[-4000:]); print(proc.stderr[-4000:])
        raise RuntimeError(f"{script} exited {proc.returncode}, expected {expect}")
    print(proc.stdout[-2000:])
    return proc

def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def latest_artifact(base: Path, prefix: str) -> Path:
    dirs = sorted((p for p in base.glob(prefix + "_*") if p.is_dir()), key=lambda p: p.stat().st_mtime)
    if not dirs:
        raise FileNotFoundError(f"no {prefix}_* artifact under {base}; did the training step run?")
    return dirs[-1]


calendar_out = WORK_DIR / "calendar"
proc = run_cli("train_forecaster.py", "--daily-csv", str(daily_csv),
               "--output-dir", str(calendar_out), "--seed", "2026")
forecaster_dir = latest_artifact(calendar_out, "forecaster")
print("forecaster artifact:", forecaster_dir)


## 2. Backtest metrics

Mean pinball loss per quantile, interval coverage, RMSE/MAE/bias; intervals stay labeled experimental/uncalibrated.

In [ ]:
import json
reports = json.loads((calendar_out / "forecast_backtest_report.json").read_text())
for name, metrics in reports.items():
    print(f"== {name} ==")
    for key, value in metrics.items():
        print(f"  {key}: {value}")
comparison = json.loads((calendar_out / "forecast_baseline_comparison.json").read_text())
print("comparison:", json.dumps(comparison, indent=2)[:1500])


## 3. Anomaly baseline sanity

Anomaly statuses are relative-to-season descriptions. `missing_coverage` days must never become numeric zero claims.

In [ ]:
import pandas as pd
anomalies = pd.read_csv(calendar_out / "anomaly_records.csv")
print(anomalies["anomaly_status"].value_counts(dropna=False).to_dict())
print(anomalies.head())
